This notebook asks, prompt by prompt, whether the model's internal signals predict what it actually does. For each harmful train prompt we know two things: what the model did (refused or complied, from the judge) and what was inside it before answering (the cached activations). If the prompts it refused carry a stronger harm signal (h) or refusal signal (r) than the ones it complied with, the weak signal is tied to the failure. If not, something else decides. Only label fields are read, never answer text.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from harm_route.judge import is_garbled, to_class

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
name = "qwen3-8b"
prompt_model = "qwen3-1.7b"            # shared prompt set, same for every model
judge = "qwen3guard-4b"
langs = ["eng_Latn", "npi_Deva", "srp_Cyrl", "srp_Latn"]
script = {l: v["script"] for l, v in yaml.safe_load(open(ROOT / "configs/languages.yaml")).items()}

A = {l: torch.load(ROOT / f"data/cache/e01/{name}/{l}_acts.pt") for l in langs}     # [lang][set]["inst"|"last"]: [N, L, d]
dirs = torch.load(ROOT / f"data/cache/e00/{name}_directions.pt")
H_LAYER, R_LAYER = dirs["h_layer"], dirs["r_layer"]
n_layers = dirs["h"].shape[0]
print(name, "layers", n_layers, "h_layer", H_LAYER, "r_layer", R_LAYER)

Turn the judge labels into groups. Refused means a clean refusal (refused, judged Safe, not garbled). Complied means it answered, harmful or benign. Refuse-then-leak and garbled answers are left out of the main comparison because they are neither clearly one nor the other; they come back in a robustness check at the end. The label file rows must line up with the cache rows (same split file, same order), which the assert checks.

In [ ]:
def load_classes(lang):
    rows = [json.loads(x) for x in open(ROOT / f"data/generations/e01_scale/{name}/{lang}/judge/{judge}/harmful_train__base.jsonl")]
    assert [r["idx"] for r in rows] == list(range(len(rows))), "label rows out of order"
    assert len(rows) == A[lang]["harmful_train"]["inst"].shape[0], "labels and cache differ in size"
    return [to_class(r, is_garbled(r["n_tokens"], r["d4"], r["script"], script[lang])) for r in rows]

cls = {l: load_classes(l) for l in langs}

def groups(lang, leak_as_complied=False):
    """1 = refused, 0 = complied, -1 = left out. Returns a [N] long tensor."""
    comply = {"comply_harmful", "comply_benign"} | ({"refuse_leak"} if leak_as_complied else set())
    return torch.tensor([1 if c == "refuse" else 0 if c in comply else -1 for c in cls[lang]])

for l in langs:
    g = groups(l)
    print(f"{l:9s} refused {(g == 1).sum():3d}  complied {(g == 0).sum():3d}  left out {(g == -1).sum():3d}")

Score every prompt along the English directions at every layer: h at the last instruction token, r at the last template token. Each layer is put on the same English scale as before (0 = average English harmless train prompt, 1 = average English harmful train prompt), so layers and languages can be compared.

In [ ]:
H = dirs["h"] / dirs["h"].norm(dim=-1, keepdim=True)              # [L, d] unit rows
R = dirs["r"][-1] / dirs["r"][-1].norm(dim=-1, keepdim=True)      # [L, d]

def proj(x, D):
    return torch.einsum("nld,ld->nl", x, D)                       # [N, L, d] x [L, d] -> [N, L], one dot product per layer

E = A["eng_Latn"]
def english_scale(key, D):
    lo = proj(E["harmless_train"][key], D).mean(0)                # [L]
    hi = proj(E["harmful_train"][key], D).mean(0)                 # [L]
    return lambda x: (proj(x, D) - lo) / (hi - lo)                # [N, L] - [L] broadcasts over prompts

scale_h, scale_r = english_scale("inst", H), english_scale("last", R)
h_score = {l: scale_h(A[l]["harmful_train"]["inst"]) for l in langs}   # [N, L]
r_score = {l: scale_r(A[l]["harmful_train"]["last"]) for l in langs}   # [N, L]
print(h_score["npi_Deva"].shape, "npi mean at h_layer", h_score["npi_Deva"][:, H_LAYER].mean().item())

AUC measures how well one number separates the two groups: pick a random refused prompt and a random complied prompt, AUC is the chance the refused one has the higher score. 0.5 means the score says nothing about behavior, 1.0 means it separates them perfectly, below 0.5 means the complied prompts score higher. It is computed by comparing every refused prompt with every complied prompt at once (broadcasting a column against a row).

In [ ]:
def auc(pos, neg):
    """P(score of a random pos > score of a random neg), ties count half. pos [n1], neg [n0]."""
    diff = pos[:, None] - neg[None, :]                            # [n1, 1] - [1, n0] -> [n1, n0], all pairs
    return ((diff > 0).float() + 0.5 * (diff == 0).float()).mean().item()

def auc_per_layer(score, g):
    return torch.tensor([auc(score[g == 1, l], score[g == 0, l]) for l in range(score.shape[1])])   # [L]

auc_h, auc_r = {}, {}
for l in langs[1:]:                                               # English refused all 130: nothing to compare
    g = groups(l)
    auc_h[l], auc_r[l] = auc_per_layer(h_score[l], g), auc_per_layer(r_score[l], g)
    print(f"{l:9s} n {(g == 1).sum()}/{(g == 0).sum()}   h AUC at L{H_LAYER} {auc_h[l][H_LAYER]:.2f} (max {auc_h[l].max():.2f} L{auc_h[l].argmax()})"
          f"   r AUC at L{R_LAYER} {auc_r[l][R_LAYER]:.2f} (max {auc_r[l].max():.2f} L{auc_r[l].argmax()})")

The same AUC at every layer. Dashed line is 0.5 (no information); the dotted vertical lines are the h and r layers. Serbian has few complied prompts, so its curves are noisy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, a, title in [(axes[0], auc_h, "h at t_inst"), (axes[1], auc_r, "r at last template token")]:
    for i, l in enumerate(langs[1:]):
        g = groups(l)
        ax.plot(a[l], color=f"C{i + 1}", label=f"{l} ({(g == 1).sum()} vs {(g == 0).sum()})")
    ax.axhline(0.5, color="k", ls="--", lw=0.8)
    for L in (H_LAYER, R_LAYER):
        ax.axvline(L, color="gray", ls=":", lw=0.8)
    ax.set_title(f"{name}: refused vs complied, {title}"); ax.set_xlabel("layer")
axes[0].set_ylabel("AUC"); axes[0].legend(); plt.tight_layout(); plt.show()

The average score of each group across layers, on the English scale. This shows not just whether the groups differ but by how much, and whether the refused prompts reach the English level (1.0) or only get partway.

In [ ]:
fig, axes = plt.subplots(2, len(langs) - 1, figsize=(14, 6), sharex=True)
for j, l in enumerate(langs[1:]):
    g = groups(l)
    for i, (s, k) in enumerate([(h_score, "h"), (r_score, "r")]):
        ax = axes[i, j]
        ax.plot(s[l][g == 1].mean(0), label="refused")
        ax.plot(s[l][g == 0].mean(0), label="complied")
        ax.axhline(1, color="k", lw=0.5); ax.axhline(0, color="gray", lw=0.5, ls="--")
        ax.set_title(f"{l}: {k}"); ax.set_xlabel("layer") if i == 1 else None
axes[0, 0].set_ylabel("English scale"); axes[1, 0].set_ylabel("English scale"); axes[0, 0].legend()
plt.tight_layout(); plt.show()

A simple confound check. If longer prompts happened to be refused more and also had larger projections, the AUC would be about length, not harm. We count instruction tokens per prompt (tokenizer only, no model) and see how well length alone predicts refusal, and how strongly length correlates with the h score.

In [ ]:
from harm_route.models import load_tokenizer, model_config
from harm_route.prompts import find_positions

tok, kw = load_tokenizer(name), model_config(name)["template_kwargs"]
for l in langs[1:]:
    prompts = json.load(open(ROOT / f"data/raw/e01/{prompt_model}/train/splits/{l}.json"))["harmful_train"]
    n_tok = torch.tensor([(p := find_positions(tok, x, kw))["t_inst"] - p["inst_start"] + 1 for x in prompts]).float()  # [N]
    g = groups(l)
    corr = torch.corrcoef(torch.stack([n_tok, h_score[l][:, H_LAYER]]))[0, 1].item()
    print(f"{l:9s} length AUC {auc(n_tok[g == 1], n_tok[g == 0]):.2f}   corr(length, h at L{H_LAYER}) {corr:+.2f}")

How sure are we? With about 70 against 40 prompts an AUC moves a lot by chance, so we resample the prompts with replacement 1000 times and take the middle 95% of AUCs (a bootstrap interval). If the interval stays above 0.5, the effect is unlikely to be noise.

In [ ]:
def boot_auc(score, g, n_boot=1000, seed=0):
    gen = torch.Generator().manual_seed(seed)
    pos, neg = score[g == 1], score[g == 0]
    out = torch.tensor([auc(pos[torch.randint(len(pos), (len(pos),), generator=gen)],
                            neg[torch.randint(len(neg), (len(neg),), generator=gen)]) for _ in range(n_boot)])
    return out.quantile(0.025).item(), out.quantile(0.975).item()

for l in langs[1:]:
    g = groups(l)
    lo_h, hi_h = boot_auc(h_score[l][:, H_LAYER], g)
    lo_r, hi_r = boot_auc(r_score[l][:, R_LAYER], g)
    print(f"{l:9s} h L{H_LAYER} AUC {auc_h[l][H_LAYER]:.2f} [{lo_h:.2f}, {hi_h:.2f}]   r L{R_LAYER} AUC {auc_r[l][R_LAYER]:.2f} [{lo_r:.2f}, {hi_r:.2f}]")

Robustness: count refuse-then-leak answers as complied (they did give harmful content). If the picture holds, the result does not depend on how we handled those borderline cases.

In [ ]:
for l in langs[1:]:
    g = groups(l, leak_as_complied=True)
    print(f"{l:9s} n {(g == 1).sum()}/{(g == 0).sum()}   h AUC L{H_LAYER} {auc(h_score[l][g == 1, H_LAYER], h_score[l][g == 0, H_LAYER]):.2f}"
          f"   r AUC L{R_LAYER} {auc(r_score[l][g == 1, R_LAYER], r_score[l][g == 0, R_LAYER]):.2f}")

Complied is really two different failures. Complied-benign may mean the model did not get what was asked; complied-harmful means it understood enough to help and did not refuse. Here every judge class gets its average h and r at the chosen layers. If the harmful compliers had a strong h but a weak r, that would be a routing failure (harm registered, refusal not triggered); if they had a weak h like the benign compliers, the harm signal itself is too weak.

In [ ]:
order = ["refuse", "refuse_leak", "comply_harmful", "comply_benign", "garbled"]
for l in langs[1:]:
    print(l)
    for k in order:
        idx = torch.tensor([i for i, c in enumerate(cls[l]) if c == k], dtype=torch.long)   # prompt indices in this class
        if len(idx) == 0:
            continue
        print(f"  {k:15s} n {len(idx):3d}   h at L{H_LAYER} {h_score[l][idx, H_LAYER].mean():5.2f}   r at L{R_LAYER} {r_score[l][idx, R_LAYER].mean():5.2f}")